In [1]:
pip install -r requirement.txt


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import copy
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

In [3]:
 #1. CONFIGURATION
# ============================================================

DATA_PATH = "heart_2022_no_nans.csv"
TARGET_COLUMN = "HadHeartAttack"

RESULTS_DIR = "results"
MODELS_DIR = "saved_models"

NUM_CLIENTS = 5
CENTRALIZED_EPOCHS = 15
FL_ROUNDS = 10
LOCAL_EPOCHS = 2

BATCH_SIZE = 256
LEARNING_RATE = 0.001
TEST_SIZE = 0.2
RANDOM_SEED = 42

DP_NOISE_STD = 0.001

In [4]:
# 2. BASIC SETUP
# ============================================================

def create_folders():
    """Create folders for saving results and trained models."""
    os.makedirs(RESULTS_DIR, exist_ok=True)
    os.makedirs(MODELS_DIR, exist_ok=True)


def set_seed(seed=42):
    """Make results more stable and reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [5]:
# 3. DATA LOADING AND PREPROCESSING
# ============================================================

def load_and_preprocess_data():
    """
    Load the Heart Disease dataset and convert it into ML-ready format.
    Categorical values are converted into numeric values using one-hot encoding.
    Numeric values are scaled using StandardScaler.
    """

    df = pd.read_csv(DATA_PATH)

    if TARGET_COLUMN not in df.columns:
        raise ValueError(f"Target column '{TARGET_COLUMN}' not found in dataset.")

    print("\nDataset loaded successfully")
    print("Dataset shape:", df.shape)

    print("\nTarget column distribution:")
    print(df[TARGET_COLUMN].value_counts())

    y = df[TARGET_COLUMN]

# Convert target column safely
# Dataset target values are usually "Yes" and "No"
    if not pd.api.types.is_numeric_dtype(y):
        y = y.astype(str).str.strip()

        print("\nUnique target values before mapping:")
        print(y.unique())

        y = y.map({
            "No": 0,
            "Yes": 1,
            "no": 0,
            "yes": 1,
            "NO": 0,
            "YES": 1,
            "0": 0,
            "1": 1
        })

# Check if any value was not converted
    if y.isnull().sum() > 0:
        print("\nUnmapped target values found:")
        print(df[TARGET_COLUMN].unique())
        raise ValueError("Some target values could not be mapped to 0 and 1.")

    y = y.astype(int)

    X = df.drop(columns=[TARGET_COLUMN])

    X = pd.get_dummies(X, drop_first=False)

    X = X.fillna(X.median(numeric_only=True))

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_SEED,
        stratify=y
    )

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    X_train_scaled = X_train_scaled.astype(np.float32)
    X_test_scaled = X_test_scaled.astype(np.float32)

    y_train = y_train.values.astype(np.float32)
    y_test = y_test.values.astype(np.float32)

    return X_train_scaled, X_test_scaled, y_train, y_test, X.columns.tolist()


In [6]:
# 4. CLIENT DATA SPLITTING
# ============================================================

def create_iid_clients(X_train, y_train, num_clients):
    """
    IID split means each client gets almost similar type of data.
    Data is randomly shuffled and divided equally among clients.
    """

    indices = np.arange(len(X_train))
    np.random.shuffle(indices)

    client_indices = np.array_split(indices, num_clients)

    clients = []

    for idx in client_indices:
        X_client = torch.tensor(X_train[idx], dtype=torch.float32)
        y_client = torch.tensor(y_train[idx], dtype=torch.float32).view(-1, 1)

        dataset = TensorDataset(X_client, y_client)
        clients.append(dataset)

    return clients


def split_indices_by_weights(indices, weights):
    """Split index list according to given weights."""
    weights = np.array(weights)
    weights = weights / weights.sum()

    split_points = (np.cumsum(weights)[:-1] * len(indices)).astype(int)

    return np.split(indices, split_points)


def create_non_iid_clients(X_train, y_train, num_clients):
    """
    non-IID split means each client has different data pattern.
    Here, some clients get more heart attack cases and some get fewer.
    """

    negative_indices = np.where(y_train == 0)[0]
    positive_indices = np.where(y_train == 1)[0]

    np.random.shuffle(negative_indices)
    np.random.shuffle(positive_indices)

    negative_weights = [0.40, 0.25, 0.18, 0.12, 0.05]
    positive_weights = [0.03, 0.07, 0.12, 0.25, 0.53]

    negative_splits = split_indices_by_weights(negative_indices, negative_weights)
    positive_splits = split_indices_by_weights(positive_indices, positive_weights)

    clients = []

    print("\nnon-IID Client Distribution:")

    for client_id in range(num_clients):
        combined_indices = np.concatenate([
            negative_splits[client_id],
            positive_splits[client_id]
        ])

        np.random.shuffle(combined_indices)

        X_client = torch.tensor(X_train[combined_indices], dtype=torch.float32)
        y_client = torch.tensor(y_train[combined_indices], dtype=torch.float32).view(-1, 1)

        dataset = TensorDataset(X_client, y_client)
        clients.append(dataset)

        total = len(y_client)
        positives = int(y_client.sum().item())
        negatives = total - positives

        print(
            f"Client {client_id + 1}: "
            f"Total={total}, No Heart Attack={negatives}, Heart Attack={positives}"
        )

    return clients


# ============================================================


In [7]:
# 5. MODEL
# ============================================================

class HeartDiseaseMLP(nn.Module):
    """
    Simple neural network for binary classification.
    MLP means Multi-Layer Perceptron.
    """

    def __init__(self, input_dim):
        super(HeartDiseaseMLP, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

In [8]:
# 6. TRAINING AND EVALUATION FUNCTIONS
# ============================================================

def calculate_pos_weight(y_train):
    """
    Calculate class imbalance weight.
    This helps the model pay more attention to the minority class.
    """

    positive_count = np.sum(y_train == 1)
    negative_count = np.sum(y_train == 0)

    if positive_count == 0:
        return 1.0

    return negative_count / positive_count


def train_one_epoch(model, data_loader, criterion, optimizer, device):
    """Train model for one epoch."""

    model.train()

    total_loss = 0

    for X_batch, y_batch in data_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch)

        loss = criterion(logits, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(data_loader)

    return average_loss


def evaluate_model(model, X_test, y_test, criterion, device):
    """Test model and calculate accuracy, precision, recall, F1 and confusion matrix."""

    model.eval()

    X_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1).to(device)

    with torch.no_grad():
        logits = model(X_tensor)
        loss = criterion(logits, y_tensor).item()
        probabilities = torch.sigmoid(logits)
        predictions = (probabilities >= 0.5).int().cpu().numpy().flatten()

    y_true = y_test.astype(int)

    metrics = {
        "loss": loss,
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1_score": f1_score(y_true, predictions, zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, predictions).tolist(),
        "classification_report": classification_report(
            y_true,
            predictions,
            target_names=["No Heart Attack", "Heart Attack"],
            zero_division=0
        )
    }

    return metrics


def print_metrics(title, metrics):
    """Print model performance clearly."""

    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)

    print(f"Loss      : {metrics['loss']:.4f}")
    print(f"Accuracy  : {metrics['accuracy'] * 100:.2f}%")
    print(f"Precision : {metrics['precision'] * 100:.2f}%")
    print(f"Recall    : {metrics['recall'] * 100:.2f}%")
    print(f"F1-score  : {metrics['f1_score'] * 100:.2f}%")

    print("\nConfusion Matrix:")
    print(np.array(metrics["confusion_matrix"]))

    print("\nClassification Report:")
    print(metrics["classification_report"])

In [9]:
# 7. CENTRALIZED TRAINING
# ============================================================

def train_centralized_model(X_train, y_train, X_test, y_test, input_dim, pos_weight, device):
    """
    Centralized model means normal ML training.
    The model gets full training data at one place.
    """

    print("\nStarting Centralized Training...")

    X_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

    train_dataset = TensorDataset(X_tensor, y_tensor)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    model = HeartDiseaseMLP(input_dim).to(device)

    pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    loss_history = []
    accuracy_history = []

    for epoch in range(1, CENTRALIZED_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)

        test_metrics = evaluate_model(model, X_test, y_test, criterion, device)

        loss_history.append(train_loss)
        accuracy_history.append(test_metrics["accuracy"])

        print(
            f"Epoch {epoch}/{CENTRALIZED_EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Test Accuracy: {test_metrics['accuracy'] * 100:.2f}%"
        )

    final_metrics = evaluate_model(model, X_test, y_test, criterion, device)

    model_path = os.path.join(MODELS_DIR, "centralized_heart_model.pth")
    torch.save(model.state_dict(), model_path)

    print(f"\nCentralized model saved at: {model_path}")

    return model, final_metrics, loss_history, accuracy_history


# ============================================================


In [10]:
# 8. FEDERATED LEARNING FUNCTIONS
# ============================================================

def get_model_size_mb(model):
    """Calculate model size in MB."""

    total_bytes = 0

    for parameter in model.state_dict().values():
        total_bytes += parameter.numel() * parameter.element_size()

    return total_bytes / (1024 * 1024)


def add_differential_privacy_noise(state_dict, noise_std):
    """
    Add small Gaussian noise to model parameters.
    This simulates Differential Privacy.
    """

    noisy_state_dict = copy.deepcopy(state_dict)

    for key in noisy_state_dict:
        noise = torch.randn_like(noisy_state_dict[key]) * noise_std
        noisy_state_dict[key] = noisy_state_dict[key] + noise

    return noisy_state_dict


def fed_avg(client_state_dicts, client_sizes):
    """
    FedAvg means Federated Averaging.
    Server combines client model weights using weighted average.
    """

    total_samples = sum(client_sizes)

    global_state_dict = copy.deepcopy(client_state_dicts[0])

    for key in global_state_dict.keys():
        global_state_dict[key] = torch.zeros_like(global_state_dict[key])

        for client_id in range(len(client_state_dicts)):
            weight = client_sizes[client_id] / total_samples
            global_state_dict[key] += client_state_dicts[client_id][key] * weight

    return global_state_dict


def train_federated_model(
    experiment_name,
    clients,
    X_test,
    y_test,
    input_dim,
    pos_weight,
    device,
    use_dp=False
):
    """
    Train model using Federated Learning.
    Each client trains locally, then server averages all client models.
    """

    print(f"\nStarting Federated Training: {experiment_name}")

    global_model = HeartDiseaseMLP(input_dim).to(device)

    pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

    round_accuracy = []
    round_loss = []

    model_size_mb = get_model_size_mb(global_model)

    for round_num in range(1, FL_ROUNDS + 1):
        client_state_dicts = []
        client_sizes = []
        client_losses = []

        print(f"\nCommunication Round {round_num}/{FL_ROUNDS}")

        for client_id, client_dataset in enumerate(clients):
            local_model = HeartDiseaseMLP(input_dim).to(device)

            local_model.load_state_dict(global_model.state_dict())

            client_loader = DataLoader(
                client_dataset,
                batch_size=BATCH_SIZE,
                shuffle=True
            )

            optimizer = optim.Adam(local_model.parameters(), lr=LEARNING_RATE)

            local_losses = []

            for _ in range(LOCAL_EPOCHS):
                local_loss = train_one_epoch(
                    local_model,
                    client_loader,
                    criterion,
                    optimizer,
                    device
                )
                local_losses.append(local_loss)

            local_state_dict = local_model.state_dict()

            if use_dp:
                local_state_dict = add_differential_privacy_noise(
                    local_state_dict,
                    DP_NOISE_STD
                )

            client_state_dicts.append(local_state_dict)
            client_sizes.append(len(client_dataset))
            client_losses.append(np.mean(local_losses))

            print(
                f"Client {client_id + 1} | "
                f"Samples: {len(client_dataset)} | "
                f"Loss: {np.mean(local_losses):.4f}"
            )

        new_global_state_dict = fed_avg(client_state_dicts, client_sizes)

        global_model.load_state_dict(new_global_state_dict)

        metrics = evaluate_model(global_model, X_test, y_test, criterion, device)

        round_accuracy.append(metrics["accuracy"])
        round_loss.append(metrics["loss"])

        print(
            f"Global Accuracy after Round {round_num}: "
            f"{metrics['accuracy'] * 100:.2f}%"
        )

    final_metrics = evaluate_model(global_model, X_test, y_test, criterion, device)

    model_filename = experiment_name.lower().replace(" ", "_") + ".pth"
    model_path = os.path.join(MODELS_DIR, model_filename)
    torch.save(global_model.state_dict(), model_path)

    total_communication_mb = 2 * NUM_CLIENTS * FL_ROUNDS * model_size_mb

    final_metrics["model_size_mb"] = model_size_mb
    final_metrics["communication_overhead_mb"] = total_communication_mb

    print(f"\nFederated model saved at: {model_path}")
    print(f"Model Size: {model_size_mb:.4f} MB")
    print(f"Communication Overhead: {total_communication_mb:.4f} MB")

    history = {
        "accuracy": round_accuracy,
        "loss": round_loss
    }

    return global_model, final_metrics, history



In [11]:
# 9. PLOTTING AND SAVING RESULTS
# ============================================================

def plot_convergence(history, title, filename):
    """Save accuracy and loss convergence plots."""

    rounds = range(1, len(history["accuracy"]) + 1)

    plt.figure()
    plt.plot(rounds, [acc * 100 for acc in history["accuracy"]], marker="o")
    plt.xlabel("Communication Rounds")
    plt.ylabel("Accuracy (%)")
    plt.title(title + " - Accuracy Convergence")
    plt.grid(True)
    plt.savefig(os.path.join(RESULTS_DIR, filename + "_accuracy.png"))
    plt.close()

    plt.figure()
    plt.plot(rounds, history["loss"], marker="o")
    plt.xlabel("Communication Rounds")
    plt.ylabel("Loss")
    plt.title(title + " - Loss Convergence")
    plt.grid(True)
    plt.savefig(os.path.join(RESULTS_DIR, filename + "_loss.png"))
    plt.close()


def save_confusion_matrix(metrics, title, filename):
    """Save confusion matrix image."""

    cm = np.array(metrics["confusion_matrix"])

    display = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["No Heart Attack", "Heart Attack"]
    )

    display.plot(values_format="d")
    plt.title(title)
    plt.savefig(os.path.join(RESULTS_DIR, filename))
    plt.close()


def save_all_results(results):
    """Save all experiment results into JSON and CSV files."""

    json_path = os.path.join(RESULTS_DIR, "all_results.json")

    with open(json_path, "w") as file:
        json.dump(results, file, indent=4)

    rows = []

    for experiment_name, metrics in results.items():
        rows.append({
            "Experiment": experiment_name,
            "Accuracy": metrics["accuracy"],
            "Precision": metrics["precision"],
            "Recall": metrics["recall"],
            "F1-score": metrics["f1_score"],
            "Loss": metrics["loss"],
            "Communication Overhead MB": metrics.get("communication_overhead_mb", 0)
        })

    df_results = pd.DataFrame(rows)

    csv_path = os.path.join(RESULTS_DIR, "accuracy_precision_recall_f1_summary.csv")
    df_results.to_csv(csv_path, index=False)

    print(f"\nResults saved at: {json_path}")
    print(f"Summary CSV saved at: {csv_path}")


def plot_accuracy_comparison(results):
    """Save bar graph comparing accuracies of all experiments."""

    experiment_names = list(results.keys())
    accuracies = [results[name]["accuracy"] * 100 for name in experiment_names]

    plt.figure(figsize=(10, 5))
    plt.bar(experiment_names, accuracies)
    plt.ylabel("Accuracy (%)")
    plt.title("Accuracy Comparison")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "accuracy_comparison.png"))
    plt.close()


# ============================================================


In [12]:
# 10. MAIN FUNCTION
# ============================================================

def main():
    """Main function controls the full assignment workflow."""

    create_folders()

    set_seed(RANDOM_SEED)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print("Using device:", device)

    X_train, X_test, y_train, y_test, feature_names = load_and_preprocess_data()

    input_dim = X_train.shape[1]

    print("\nNumber of input features after encoding:", input_dim)

    pos_weight = calculate_pos_weight(y_train)

    print("Positive class weight:", pos_weight)

    results = {}

    centralized_model, centralized_metrics, centralized_loss, centralized_acc = train_centralized_model(
        X_train,
        y_train,
        X_test,
        y_test,
        input_dim,
        pos_weight,
        device
    )

    results["Centralized Training"] = centralized_metrics

    print_metrics("Centralized Training Results", centralized_metrics)

    save_confusion_matrix(
        centralized_metrics,
        "Centralized Training Confusion Matrix",
        "centralized_confusion_matrix.png"
    )

    iid_clients = create_iid_clients(X_train, y_train, NUM_CLIENTS)

    fl_iid_model, fl_iid_metrics, fl_iid_history = train_federated_model(
        "FL IID",
        iid_clients,
        X_test,
        y_test,
        input_dim,
        pos_weight,
        device,
        use_dp=False
    )

    results["FL-IID"] = fl_iid_metrics

    print_metrics("FL-IID Results", fl_iid_metrics)

    plot_convergence(fl_iid_history, "FL-IID", "fl_iid")
    save_confusion_matrix(fl_iid_metrics, "FL-IID Confusion Matrix", "fl_iid_confusion_matrix.png")

    non_iid_clients = create_non_iid_clients(X_train, y_train, NUM_CLIENTS)

    fl_non_iid_model, fl_non_iid_metrics, fl_non_iid_history = train_federated_model(
        "FL non IID",
        non_iid_clients,
        X_test,
        y_test,
        input_dim,
        pos_weight,
        device,
        use_dp=False
    )

    results["FL-non-IID"] = fl_non_iid_metrics

    print_metrics("FL-non-IID Results", fl_non_iid_metrics)

    plot_convergence(fl_non_iid_history, "FL-non-IID", "fl_non_iid")
    save_confusion_matrix(fl_non_iid_metrics, "FL-non-IID Confusion Matrix", "fl_non_iid_confusion_matrix.png")

    fl_dp_model, fl_dp_metrics, fl_dp_history = train_federated_model(
        "FL Differential Privacy",
        non_iid_clients,
        X_test,
        y_test,
        input_dim,
        pos_weight,
        device,
        use_dp=True
    )

    results["FL + Differential Privacy"] = fl_dp_metrics

    print_metrics("FL + Differential Privacy Results", fl_dp_metrics)

    plot_convergence(fl_dp_history, "FL + Differential Privacy", "fl_dp")
    save_confusion_matrix(fl_dp_metrics, "FL + DP Confusion Matrix", "fl_dp_confusion_matrix.png")

    plot_accuracy_comparison(results)

    save_all_results(results)

    print("\nAssignment completed successfully.")
    print("Check the 'results' folder for plots and metrics.")
    print("Check the 'saved_models' folder for trained models.")





In [13]:
if __name__ == "__main__":
    main()

Using device: cuda

Dataset loaded successfully
Dataset shape: (246022, 40)

Target column distribution:
HadHeartAttack
No     232587
Yes     13435
Name: count, dtype: int64

Unique target values before mapping:
<StringArray>
['No', 'Yes']
Length: 2, dtype: str

Number of input features after encoding: 154
Positive class weight: 17.3119650167473

Starting Centralized Training...
Epoch 1/15 | Train Loss: 0.8329 | Test Accuracy: 79.77%
Epoch 2/15 | Train Loss: 0.7856 | Test Accuracy: 79.23%
Epoch 3/15 | Train Loss: 0.7800 | Test Accuracy: 79.80%
Epoch 4/15 | Train Loss: 0.7723 | Test Accuracy: 80.49%
Epoch 5/15 | Train Loss: 0.7657 | Test Accuracy: 80.41%
Epoch 6/15 | Train Loss: 0.7573 | Test Accuracy: 80.17%
Epoch 7/15 | Train Loss: 0.7533 | Test Accuracy: 80.66%
Epoch 8/15 | Train Loss: 0.7467 | Test Accuracy: 79.51%
Epoch 9/15 | Train Loss: 0.7396 | Test Accuracy: 80.39%
Epoch 10/15 | Train Loss: 0.7355 | Test Accuracy: 80.83%
Epoch 11/15 | Train Loss: 0.7286 | Test Accuracy: 78.03%
